# Comparative Analysis of West African Economic Growth
### Objective
This project explores the GDP per capita trends of **Burkina Faso** and **Côte d'Ivoire** using the World Bank API. 

### Personal Context
As an aspiring engineer with experience in the West African digital sector (notably at Orange Burkina Faso and Consultech Abidjan), I have developed this tool to visualize regional economic trajectories through data engineering and mathematical smoothing.

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import requests

# Optimization: Using a dictionary to map ISO codes for better readability
COUNTRIES = {'BFA': 'Burkina Faso', 'CIV': 'Côte d\'Ivoire'}

## 1. Data Acquisition Pipeline
We fetch 'GDP per capita (current US$)' for both Burkina Faso and Côte d'Ivoire.

In [6]:
def get_cleaned_data(country_code):
    url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/NY.GDP.PCAP.CD?format=json&per_page=100"
    try:
        response = requests.get(url)
        data = response.json()[1]
        df = pd.DataFrame(data)
        df = df[['date', 'value']].rename(columns={'date': 'Year', 'value': 'GDP_USD'})
        df['Year'] = df['Year'].astype(int)
        df['Country'] = COUNTRIES[country_code]
        return df.dropna().sort_values('Year')
    except Exception as e:
        return pd.DataFrame()

# Combine data for both countries [cite: 30]
df_bfa = get_cleaned_data('BFA')
df_civ = get_cleaned_data('CIV')
df_final = pd.concat([df_bfa, df_civ])

df_final.sample(5)

,Year,GDP_USD,Country
31,1994,581.888469,Côte d'Ivoire
53,1972,99.633581,Burkina Faso
59,1966,221.786392,Côte d'Ivoire
15,2010,1553.548870,Côte d'Ivoire
41,1984,187.832224,Burkina Faso


## 2. Statistical Smoothing (Moving Average)
To compare the growth stability of the two economies, we apply a 5-year Simple Moving Average (SMA). 
This demonstrates the implementation of basic signal processing in economic datasets.

In [7]:
# Apply smoothing per country group
df_final['SMA_5'] = df_final.groupby('Country')['GDP_USD'].transform(lambda x: x.rolling(window=5).mean())

# Identify the growth rate (Mathematical Engineering touch)
df_final['Yearly_Growth_Rate'] = df_final.groupby('Country')['GDP_USD'].pct_change() * 100

## 3. Comparative Visualization
Visualizing the evolution of GDP per capita and the smoothed trend for both nations.

In [8]:
fig = px.line(df_final, x='Year', y='SMA_5', color='Country',
              title='5-Year Smoothed GDP Per Capita: Burkina Faso vs. Côte d\'Ivoire',
              labels={'SMA_5': 'GDP per Capita (USD) - Smoothed', 'Year': 'Year'},
              template='plotly_white')

fig.show()